In [1]:
import sys, os, shutil, glob

# Copy entire project to writable location
src = '/kaggle/input/datasets/ericazhang0127/aml-gnn-code/aml-gnn-detection'
dst = '/kaggle/working/aml-gnn-detection'

if not os.path.exists(dst):
    shutil.copytree(src, dst)
    print("Project copied to working directory")
else:
    print("Project already in working directory")

# Set path to writable copy
sys.path.insert(0, dst)

# Create data/raw and copy CSVs into it
raw_dir = f'{dst}/data/raw'
os.makedirs(raw_dir, exist_ok=True)

for f in glob.glob('/kaggle/input/**/*.csv', recursive=True):
    if 'Trans' in f:
        shutil.copy(f, raw_dir)
        print(f"Copied: {f.split('/')[-1]}")

print("\nAll set — project is at:", dst)

Project already in working directory
Copied: LI-Small_Trans.csv
Copied: HI-Small_Trans.csv

All set — project is at: /kaggle/working/aml-gnn-detection


In [2]:
from src.data.loader import load_raw
print("Import OK")

Import OK


In [3]:
!pip install torch-geometric -q
print("torch-geometric installed")

torch-geometric installed


In [4]:
from src.data.loader import load_multiple

raw_dir = '/kaggle/working/aml-gnn-detection/data/raw'
df = load_multiple(raw_dir)
print(f"Loaded {len(df):,} transactions")

591212 self-loop transactions (from_account == to_account). These will appear as self-edges in the graph.
9 fully duplicate rows detected.
804477 self-loop transactions (from_account == to_account). These will appear as self-edges in the graph.
8 fully duplicate rows detected.
Dropped 17 duplicate rows after concatenation.


Loaded 12,002,377 transactions


In [5]:
from src.data.graph_builder import build_graph
from src.utils.seed import set_seed

set_seed(42)

data, stats = build_graph(
    df,
    train=0.70,
    val=0.15,
    test=0.15,
    seed=42,
    verbose=True,
)
print("Graph was built successfully")


╔══════════════════════════════════════════════════════╗
║  Graph Construction — Summary                        ║
╠══════════════════════════════════════════════════════╣
  Nodes (accounts)  :    1,208,958
  Edges (txns)      :   12,002,377
  Node features     :           12
  Edge features     :            4
──────────────────────────────────────────────────────
  Train nodes       :      846,270  (illicit 0.96%)
  Val   nodes       :      181,344  (illicit 0.96%)
  Test  nodes       :      181,344  (illicit 0.96%)
──────────────────────────────────────────────────────
  Overall illicit   :        0.96%
╚══════════════════════════════════════════════════════╝

Graph was built successfully


In [19]:
from src.data.transforms import fit_transform

data, transformer = fit_transform(data)
print("Transforms applied")

Transforms applied


In [20]:
from src.models.graphsage import build_model_from_cfg
from src.training.loss import build_loss

cfg = {
    "model": {
        "hidden_dim": 128,
        "out_dim": 1,
        "num_layers": 3,
        "aggregator": "mean",
        "dropout": 0.3,
        "batch_norm": True,
        "residual": True,
    },
    "data":     {"pos_weight": "auto"},
    "training": {"loss": "weighted_bce"},
}

model = build_model_from_cfg(cfg["model"], in_dim=data.num_node_features)
print(f"Model parameters: {model.count_parameters():,}")

criterion = build_loss(cfg, train_labels=data.y, train_mask=data.train_mask)
print(f"Loss: {type(criterion).__name__}")

Model parameters: 71,425
Loss: WeightedBCELoss


In [ ]:
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.10.0+cpu.html -q
print("torch-scatter installed")

In [ ]:
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.10.0+cpu.html -q
print("torch-sparse installed")

In [8]:
import torch
import torch_scatter
import torch_sparse
import torch_geometric

print("torch:", torch.__version__)
print("torch_scatter:", torch_scatter.__version__)
print("torch_sparse:", torch_sparse.__version__)
print("torch_geometric:", torch_geometric.__version__)
print("\nAll dependencies are OK")

torch: 2.10.0+cu128
torch_scatter: 2.1.2+pt210cpu
torch_sparse: 0.6.18+pt210cpu
torch_geometric: 2.7.0

All dependencies are OK


In [22]:
print(torch.cuda.is_available())        # should print True
print(torch.cuda.get_device_name(0))    # should print GPU name

True
Tesla T4


In [25]:
!pip install pyg-lib -f https://data.pyg.org/whl/torch-2.10.0+cu128.html -q
print("Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 41.8 MB/s eta 0:00:0000:0100:01
Done


In [27]:
import pyg_lib

In [28]:
from src.training.trainer import Trainer

train_cfg = {
    "training": {
        "epochs": 50,
        "lr": 0.001,
        "weight_decay": 0.0001,
        "batch_size": 1024,
        "num_neighbors": [25, 10, 5],
        "patience": 10,
        "min_delta": 0.0001,
        "gradient_clip": 1.0,
        "scheduler": {"type": "cosine", "warmup_epochs": 5},
    },
    "logging": {
        "save_best_only": True,
        "log_every_n_steps": 1,
        "use_wandb": False,
    },
    "paths": {
        "checkpoints": "/kaggle/working/checkpoints",
        "reports":     "/kaggle/working/reports",
    },
}

trainer = Trainer(train_cfg, data, model, criterion, device="cuda")
history = trainer.fit()

print(f"\nBest epoch  : {history.best_epoch}")
print(f"Best AUC-PR : {history.best_auc_pr:.4f}")


Best epoch  : 50
Best AUC-PR : 0.1385


In [30]:
import torch
import numpy as np
from src.evaluation.metrics import evaluate

model.eval()
with torch.no_grad():
    probs = model.predict_proba(data.x, data.edge_index).cpu().numpy()

test_mask   = data.test_mask.cpu().numpy()
test_probs  = probs[test_mask]
test_labels = data.y.cpu().numpy()[test_mask]

report = evaluate(
    labels    = test_labels,
    probs     = test_probs,
    threshold = 0.5,
    run_id    = "kaggle_run_01",
    split     = "test",
    verbose   = True,
)


╔══════════════════════════════════════════════════════════╗
║  Evaluation Report  [ TEST ]  run: kaggle_run_01        ║
╠══════════════════════════════════════════════════════════╣
  Nodes evaluated  :    181,344
  Illicit (actual) :      1,749  (0.964%)
──────────────────────────────────────────────────────────
  AUC-PR           :     0.1417   ← PRIMARY metric
  AUC-ROC          :     0.9035
──────────────────────────────────────────────────────────
  Threshold        :     0.5000
  F1               :     0.0787
  Precision        :     0.0413
  Recall           :     0.8268
  FPR              :     0.1869
  FNR (miss rate)  :     0.1732
──────────────────────────────────────────────────────────
  Optimal threshold:     0.9182  (max F1)
  Optimal F1       :     0.1794
──────────────────────────────────────────────────────────
  Confusion matrix :  Pred licit  Pred illicit
    Actual licit   :  TN=146,036   FP=33,559
    Actual illicit :  FN=303       TP=1,446
╚═════════════════════